# SMARTRENT — ไฟล์ที่ 2: ตรวจสอบและฝึกโมเดลพื้นฐาน Phase 2 ซ้ำ (18 ตัวแปรห้อง)

> **ลำดับสมุดโน้ตของโปรเจกต์ (3 ไฟล์ต่อเนื่องกัน):**
> 1. `1_Data_Prep.ipynb` — เตรียมข้อมูลจากไฟล์ดิบ 202306.xlsx และคัดเลือกโมเดล
> 2. **`2_Phase2_Model.ipynb`** *(ไฟล์นี้)* — โหลดชุดข้อมูล CSV ที่เตรียมไว้แล้วมาตรวจสอบและรันโมเดลฐานของ Phase 2 (18 ตัวแปรห้อง)
> 3. `3_Phase3_Model.ipynb` — เพิ่มข้อมูลโครงการ +6 ตัวแปร (รวม 24 ตัวแปร) และเปรียบเทียบพัฒนาการกับ Phase 2

**ไฟล์นี้มีไว้ทำอะไร?**  
ใช้ตรวจสอบว่าโมเดลพื้นฐานจาก Phase 2 (`models/phase2_ridge_rent_model.joblib`) ที่รับเฉพาะ **ข้อมูลภายในห้อง 18 ตัวแปร** ทำงานได้ถูกต้อง ตรงตามผลทดสอบ `RMSE = 29,879.92 บาท` และเก็บไว้เป็นเกณฑ์มาตรฐาน (Baseline) สำหรับเทียบกับ Phase 3

## ขั้นตอนที่ 1: เตรียมไลบรารีและโหลดข้อมูล
- ติดตั้งไลบรารีที่จำเป็นด้วย `pip install -r requirements.txt` (ถ้ายังไม่ได้ติดตั้ง)
- โค้ดด้านล่างจะเชื่อมต่อฟังก์ชันจาก `python_code/pipeline.py` แล้วโหลดข้อมูล `train` (4,217 ห้อง), `validation` (1,408 ห้อง), และ `test` (800 ห้อง) ขึ้นมาพร้อมใช้งาน

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data' / 'train.csv').exists():
    if (ROOT.parent / 'data' / 'train.csv').exists():
        ROOT = ROOT.parent
    else:
        candidates = list(ROOT.glob('*/data/train.csv'))
        if len(candidates) != 1:
            raise FileNotFoundError('Set ROOT to the smartrent-condo-rent-prediction folder')
        ROOT = candidates[0].parents[1]

sys.path.insert(0, str(ROOT / 'python_code'))
from pipeline import load_data, verify_saved, reproduce_ridge, reproduce_all, FEATURES
import pandas as pd

train, validation, test, train_plus_val = load_data(ROOT)
print('Train:', len(train), 'Validation:', len(validation), 'Test:', len(test), 'Train+Val:', len(train_plus_val))

Train: 4217 Validation: 1408 Test: 800 Train+Val: 5625


## ขั้นตอนที่ 2: ตรวจสอบรายชื่อตัวแปรทั้ง 18 ตัวของโมเดล Phase 2
- **ตัวแปรทำนาย (18 Features):** โครงสร้างห้อง 5 ตัว (`log_size`, `floor_no`, `bedrooms`, `unit_type`, `is_duplex`) + สิ่งอำนวยความสะดวกภายในห้อง 13 ตัว (ตัวแปรนับรวม `n_reported_room_amenities` และของใช้ในห้อง 12 รายการ)
- **ตัวแปรเป้าหมาย (Target):** `rental` (ค่าเช่ารายเดือน หน่วยบาท/เดือน)
- ส่วน `room_id`, `project_id`, `condo_link` ใช้สำหรับอ้างอิงและแบ่งกลุ่มข้อมูลเท่านั้น ไม่ได้ใส่เข้าสมการทำนาย

In [2]:
print('Model inputs:',len(FEATURES))
print(FEATURES)
print(train[FEATURES+['rental']].head().to_string(index=False))

Model inputs: 18
['log_size', 'floor_no', 'bedrooms', 'is_duplex', 'n_reported_room_amenities', 'air_conditioner', 'digital_door_lock', 'furnished', 'hot_tub', 'in_room_wifi', 'kitchen_hood', 'kitchen_stove', 'phone', 'refrigerator', 'tv', 'washer', 'water_heater', 'unit_type']
 log_size  floor_no  bedrooms  is_duplex  n_reported_room_amenities  air_conditioner  digital_door_lock  furnished  hot_tub  in_room_wifi  kitchen_hood  kitchen_stove  phone  refrigerator  tv  washer  water_heater unit_type  rental
 3.806662      27.0         1          0                          3                1                  0          1        0             0             0              0      0             0   0       0             1 1 Bedroom   10000
 3.828641       8.0         1          0                          3                1                  0          1        0             0             0              0      0             0   0       0             1 1 Bedroom   11000
 3.828641      33.0      

## ขั้นตอนที่ 3: โหลดไฟล์โมเดล Phase 2 มาตรวจผลบน Test Set 800 ห้อง
เรียกฟังก์ชัน `verify_saved(ROOT)` เพื่อโหลดไฟล์ `models/phase2_ridge_rent_model.joblib` มาทำนายชุด `test` 800 ห้อง และตรวจสอบความถูกต้องตรงกับตารางรายงาน

In [3]:
print(verify_saved(ROOT).to_string(index=False))

                            model      MAE_THB     RMSE_THB       R2  MAPE_pct  within20_pct  project_macro_MAE_THB
          Saved Ridge (Phase 2) 13866.355268 29879.921466 0.708932 34.100326        31.375           20341.924585


## ขั้นตอนที่ 4: ทดลองฝึกโมเดล Ridge Regression (Phase 2) ซ้ำตั้งแต่ต้น
สั่งเทรนโมเดล Ridge Regression (`alpha=10`) ด้วยข้อมูล 18 ตัวแปรขึ้นมาใหม่แล้วบันทึกผลชั่วคราวไว้ใน `my_results/ridge` โดยไม่ทับไฟล์โมเดลหลัก

In [4]:
print(reproduce_ridge(ROOT,ROOT/'my_results/ridge').to_string(index=False))

     stage model      MAE_THB     RMSE_THB       R2  MAPE_pct  within20_pct  project_macro_MAE_THB
Validation Ridge  9018.112274 14307.842020 0.595859 31.180626     38.210227           15745.943813
      Test Ridge 13866.355268 29879.921466 0.708932 34.100326     31.375000           20341.924585


## ขั้นตอนที่ 5 (ทางเลือกเพิ่มเติม): รันเปรียบเทียบทั้ง 4 อัลกอริทึมของ Phase 2
หากต้องการทดสอบเทรนเปรียบเทียบทั้ง `Median baseline`, `Ridge`, `Random Forest`, และ `Gradient Boosting` พร้อมกันทั้งหมด สามารถรันคำสั่งด้านล่างได้:
```python
print(reproduce_all(ROOT, ROOT / "my_results/all_models").to_string(index=False))
```

## สรุปผลของ Phase 2 และขั้นถัดไป
- โมเดล Phase 2 ที่ใช้เฉพาะข้อมูลภายในห้อง 18 ตัวแปรได้ผลบน Test Set 800 ห้องคือ `MAE = 13,866.36 บาท`, `RMSE = 29,879.92 บาท`, `R² = 0.7089`, และทายใกล้เคียง ±20% ได้ `31.38%`
- ขั้นตอนการเตรียมข้อมูลอัตโนมัติ (Median Imputation, StandardScaler, One-Hot Encoding) อยู่ใน `python_code/pipeline.py` เรียบร้อยแล้ว
- **เปิดดูการอัปเกรดโมเดลใน Phase 3 ต่อได้ที่ไฟล์:** `notebooks/3_Phase3_Model.ipynb`